# Somoclu SOM Summarizer Demo - KiDS-Legacy gold-weight calibration

**Original authors:** Ziang Yan, Sam Schmidt


**References:**
- Stölzner et al. 2025 (KiDS-Legacy redshift calibration), [arXiv:2503.19440](https://arxiv.org/abs/2503.19440)
- Wright et al. 2025 (KiDS-Legacy cosmic shear), [arXiv:2503.19441](https://arxiv.org/abs/2503.19441)

This notebook shows a demonstration of the use of the `SOMocluSummarizer` summarization module, modified to follow the **KiDS-Legacy** SOM calibration strategy described in Stölzner et al. 2025 ([arXiv:2503.19440](https://arxiv.org/abs/2503.19440)). Algorithmically, this module is not very different from the NZDir estimator/summarizer.  NZDir operates by finding neighboring photometric points around spectroscopic objects.  SOMocluSummarizer takes a large training set of data in the `Inform_SOMocluUmmarizer` stage and trains a self-organized map (SOM) (using code from the `somoclu` package available at: https://github.com/peterwittek/somoclu/).  Once the SOM is set up, the "best match unit" are determined for both the photometric/unknown data and a set of spectroscopic data with known redshifts.  For each SOM cell, the algorithm constructs a histogram using the spectroscopic members mapped to that cell, and weights these by the number of photometric galaxies in that cell.  Both the photometric and spectroscopic datasets can also employ an optional weight per-galaxy. <br>

The KiDS-1000 analysis (see the SOM appendices of [arXiv:1909.09632](http://arxiv.org/abs/1909.09632)) flagged SOM cells that contained photometric data but no calibrating spec-z as "non-gold", and removed the corresponding photometric galaxies from the tomographic sample with a hard cut. Because a single SOM training is stochastic (it depends on the photometric noise realization and the random initialization/ordering of the training), this hard "gold class" cut could flicker for galaxies that live close to the gold/non-gold boundary in colour-space, injecting extra scatter into the calibration.

**KiDS-Legacy modification (this notebook):** following Stölzner et al. 2025, Sect. 4 and Fig. 3 of [arXiv:2503.19440](https://arxiv.org/pdf/2503.19440), instead of training a single SOM and applying a hard gold/non-gold cut, we train $N_\mathrm{repl}$ independent SOM realizations. For each realization $j$, every photometric galaxy $i$ is given a binary "gold class" $g_{i,j}\in\{0,1\}$ depending on whether its best-matching SOM cell contains any calibrating spec-z galaxies in that realization. The **gold weight** of each galaxy is then defined as the fraction of realizations in which it was classified gold:

$$W^{\rm gold}_i = \frac{1}{N_{\rm repl}}\sum_{j=1}^{N_{\rm repl}} g_{i,j}$$

This continuous weight (0 to 1) replaces the hard gold-class cut: it is added to the photometric catalog as a per-galaxy column, and it is used as a per-galaxy weight (`phot_weightcol`) when the SOM summarizer builds the final redshift distribution, down-weighting (rather than discarding) galaxies that only sometimes fall into non-gold cells.

As with the other RAIL summarizers, we also bootstrap the spectroscopic sample and return N bootstraps in an ensemble, along with a single fiducial N(z) estimate. All of the original notebook's redshift-distribution evaluation and plotting cells are kept below, now applied to the gold-weighted calibration.

Let's set up our dependencies:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import rail
import os
import qp
import tables_io
from rail.core.data import TableHandle, Hdf5Handle
from rail.core.stage import RailStage
from rail.utils.path_utils import find_rail_file


First, let's grab some data files.  For the SOM, we will want to train on a fairly large, representative set that encompasses all of our expected data.  We'll grab a larger data file than we typically use in our demos to ensure that we construct a meaningful SOM.

This data consists of ~150,000 galaxies from a single healpix pixel of the comsoDC2 truth catalog with mock 10-year magnitude errors added.  It is cut at a relatively bright i<23.5 magnitudes in order to concentrate on galaxies with particularly high S/N rates.

In [ ]:
training_file = "./healpix_10326_bright_data.hdf5"

if not os.path.exists(training_file):
  os.system('curl -O https://portal.nersc.gov/cfs/lsst/PZ/healpix_10326_bright_data.hdf5')

In [ ]:
# way to get big data file
training_data = TableHandle("training_data",path=training_file)

Now, let's set up the inform stage for our summarizer.  We import everything from `somoclu_som` with a wildcard import (as in the original notebook) and additionally grab the private `_computemagcolordata` helper, which we will reuse below to build the colour features for the gold-weight calculation exactly as the RAIL stages do internally.

In [ ]:
from rail.estimation.algos.somoclu_som import *
from rail.estimation.algos.somoclu_som import _computemagcolordata

We need to define all of our necessary initialization params, which includes the following:
- `name` (str): the name of our estimator, as utilized by ceci
- `model` (str): the name for the model file containing the SOM and associated parameters that will be written by this stage
- `hdf5_groupname` (str): name of the hdf5 group (if any) where the photometric data resides in the training file
- `n_rows` (int): the number of dimensions in the y-direction for our 2D SOM
- `n_columns` (int): the number of dimensions in the x-direction for our 2D SOM
- `gridtype` (str): the parameter that specifies the grid form of the nodes. Options: `rectangular`(default) and `hexagonal`.
- `initialization` (str): the parameter specifying the method of initializing the SOM. Options: `pca`: principal componant analysis (default); `random`: randomly initialize the SOM.
- `maptype` (str): the parameter specifying the map topology. Options: `planar`(default) and `toroid`.
- `n_epochs` (int): the number of iteration steps during SOM training.  SOMs can take a while to converge, so we will use a fairly large number of iterations.
- `std_coeff` (float): the "radius" of how far to spread changes in the SOM
- `som_learning_rate` (float): a number between 0 and 1 that controls how quickly the weighting function decreases.  SOM's are not guaranteed to converge mathematically, and so this parameter tunes how the response drops per iteration.  A typical values we might use might be between 0.5 and 0.75.
- `column_usage` (str):  this value determines what values will be used to construct the SOM, valid choices are `colors`, `magandcolors`, and `columns`.  If set to `colors`, the code will take adjacent columns as specified in `usecols` to construct colors and use those as SOM inputs.  If set to `magandcolors` it will use the single column specfied by `ref_column_name` and the aforementioned colors to construct the SOM.  If set to `columns` then it will simply take each of the columns in `usecols` with no modification.  So, if a user wants to use K magnitudes and L colors, they can precompute the colors and specify all names in `usecols`.  NOTE: accompanying `usecols` you must have a `nondetect_val` dictionary that lists the replacement values for any non-detection-valued entries for each column, see the code for an example dictionary.  We will set `column_usage` to colors and use only colors in this example notebook.

In [ ]:
dim = 45
grid_type = 'hexagonal'

## A dedicated SOM for the gold-class (KiDS-1000-style) comparison

Before setting up the KiDS-Legacy multi-realization ensemble, we first train a single, dedicated SOM to serve as the traditional KiDS-1000-style gold-class comparison: the same `inform_dict` values used throughout this notebook (only the output filename differs, to keep its model file separate from the ensemble's), the same `training_data` `TableHandle` (read directly from `training_file`, not a bootstrap resample), and the standard two-step `.inform()` / `.model` pattern. We train it now, first, before anything else in this notebook touches SOM training, so its result cannot be affected by the separate, bootstrap-resampled ensemble trained below for the gold weight.

Critically, further down we will summarize with this model **without** passing any `phot_weightcol`, rather than reimplementing the gold/non-gold cell exclusion by hand. `SOMocluSummarizer` already excludes photometric galaxies in SOM cells with no calibrating spec-z from the N(z) sum internally (that is what the "gold class" cut *is*), so relying on that existing, well-tested logic is the only way to guarantee an exact match rather than a hand-rolled approximation of it.

In [ ]:
goldclass_inform_dict = dict(model='output_SOMoclu_model_goldclass.pkl',
                   hdf5_groupname='photometry',
                   n_rows=dim, n_columns=dim,
                   gridtype = grid_type,
                   maptype = 'toroid',
                   n_epochs=30,
                   std_coeff=12.0, som_learning_rate=0.75,
                   column_usage='colors')

inform_som_goldclass = SOMocluInformer.make_stage(name='inform_som_goldclass', **goldclass_inform_dict)

In [ ]:
%%time
inform_som_goldclass.inform(training_data)

In [ ]:
model_goldclass = inform_som_goldclass.model

## KiDS-Legacy gold weight: training multiple SOM realizations

**KiDS-Legacy addition:** `N_REALIZATIONS` sets $N_\mathrm{repl}$, the number of independent SOM trainings used to compute the gold weight (Stölzner et al. 2025 use $N_\mathrm{repl}=10$). Following the paper, every realization is trained on the **same** training catalog -- we do **not** bootstrap-resample it. What differs between realizations is only the SOM's random codebook initialization (see the note below for how this is actually achieved with the currently installed `rail_som`). Reduce `N_REALIZATIONS` (e.g. to 3) for a quicker test run.

In [ ]:
# base configuration shared by every SOM realization; `model` and `seed`
# are set individually for each realization further down
base_inform_dict = dict(hdf5_groupname='photometry',
                   n_rows=dim, n_columns=dim,
                   gridtype = grid_type,
                   maptype = 'toroid',
                   initialization = 'random',
                   n_epochs=30,
                   std_coeff=12.0, som_learning_rate=0.75,
                   column_usage='colors')

N_REALIZATIONS = 20  # N_repl in Stolzner et al. 2025 (arXiv:2503.19440)

Let's run the inform stage `N_REALIZATIONS` times, every time on the exact same `training_data`. Each run writes out its own model file (e.g. `output_SOMoclu_model_real0.pkl`, `output_SOMoclu_model_real1.pkl`, ...) and we keep the resulting model dictionaries in the `som_models` list for later use.

**How the realizations actually differ, given identical training data:** we set `initialization='random'` in `base_inform_dict`, but the currently installed `rail_som` (`SOMocluInformer.run()`) hardcodes `initialization='pca'` inside its call to `Somoclu(...)`, silently ignoring this config value -- so every realization is, in fact, PCA-initialized. This does **not** make the realizations identical, however: somoclu's PCA initialization (`_pca_init()` in `somoclu/train.py`) computes its principal components with `sklearn.decomposition.PCA(svd_solver='randomized')`, and since no `random_state` is fixed there, that randomized SVD solver draws on **numpy's global random state**. By calling `np.random.seed(seed_j)` with a different seed before each `.inform()` call, we make each realization's PCA-based codebook initialization genuinely different -- and therefore each realization converges to a different final SOM -- even though every realization sees exactly the same training catalog. This matches Stölzner et al. 2025's description of the realizations differing only in the (stochastic) SOM training, not in the underlying calibration sample.

**NOTE for those using M1 Macs:** you may get an error like `wrap_train not found` when running the inform stage in the cell just below here.  If so, this can be solved by reinstalling somoclu from conda rather than pip with the command:
```
conda install -c conda-forge somoclu
```

In [ ]:
%%time
full_training_dict = tables_io.read(training_file)

som_models = []
rng_master = np.random.default_rng(42)

for j in range(N_REALIZATIONS):
    seed_j = int(rng_master.integers(0, 1_000_000))

    inform_dict_j = dict(base_inform_dict)
    inform_dict_j['model'] = f'output_SOMoclu_model_real{j}.pkl'
    inform_dict_j['seed'] = seed_j

    # Every realization trains on the SAME (full, non-bootstrapped) training_data;
    # only the codebook initialization differs between realizations. somoclu's PCA
    # codebook init uses sklearn's PCA(svd_solver='randomized'), which (with no
    # random_state fixed) draws on numpy's global RNG state -- so seeding it
    # differently before each .inform() call gives each realization a genuinely
    # different, random starting codebook even though the training data is identical.
    np.random.seed(seed_j)

    inform_som_j = SOMocluInformer.make_stage(name=f'inform_som_real{j}', **inform_dict_j)
    inform_som_j.inform(training_data)
    som_models.append(inform_som_j.model)
    print(f"Trained SOM realization {j + 1}/{N_REALIZATIONS} (same training set, seed={seed_j})")

Running the training loop took several times longer than training a single SOM, since we train `N_REALIZATIONS` independent maps.  Remember that in a production KiDS-Legacy-like analysis, these SOMs are trained once and reused for the estimate/summarize stage many times without needing to be re-run.

We designate the **first** realization as our fiducial SOM: it is used below for the illustrative occupation/mean-redshift maps, for the Figure 3-style gold-weight visualization, and as the SOM passed to `SOMocluSummarizer` to build the gold-weighted final N(z).  The gold weight itself is computed by combining information from *all* `N_REALIZATIONS` SOMs (see the Gold weight section below). The gold-**class** (hard-cut) comparison further down uses its own, separately-trained SOM rather than this ensemble -- see the Gold weight section.

In [ ]:
model = som_models[0]
SOM = model['som']
usecols = model['usecols']
ref_column_name = model['ref_column']
column_usage = model['column_usage']
n_rows = model['n_rows']
n_columns = model['n_columns']

To visualize our fiducial SOM, let's calculate the cell occupation of our training sample, as well as the mean redshift of the galaxies in each cell.  The SOM took colors as inputs, so we will need to construct the colors for our training set galaxies:

In [ ]:
training_data.data = dict(photometry=full_training_dict['photometry'])
bands = ['u','g','r','i','z','y']
bandnames = [f"mag_{band}_lsst" for band in bands]
ngal = len(training_data.data['photometry']['mag_i_lsst'])
colors = np.zeros([5, ngal])
for i in range(5):
    colors[i] = training_data.data['photometry'][bandnames[i]] - training_data.data['photometry'][bandnames[i+1]]

We can calculate the best SOM cell using the get_bmus() function defined in somoclu_som.py, which will return the 2D SOM coordinates for each galaxy, and then use these for our visualizations (this step might take a while):

In [ ]:
bmu_coordinates = get_bmus(SOM, colors.T).T

In [ ]:
meanszs = np.zeros_like(SOM.umatrix)
cellocc = np.zeros_like(SOM.umatrix)

for i in range(training_data.data['photometry']['redshift'].size):
    bmu_x, bmu_y = bmu_coordinates.T[i]
    meanszs[bmu_x, bmu_y] += training_data.data['photometry']['redshift'][i]
    cellocc[bmu_x, bmu_y] += 1
meanszs /= cellocc

Here is the cell occupation distribution:

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12,12))
plot_som(ax, cellocc.T, grid_type=grid_type, colormap=cm.coolwarm, cbar_name='cell occupation')

And here is the mean redshift per cell:

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12,12))
plot_som(ax, meanszs.T, grid_type=grid_type, colormap=cm.coolwarm, cbar_name='mean redshift')

Note that there is spatial correlation between redshift and cell position, which is good, this is showing how there are gradual changes in redshift between similarly-colored galaxies (and sometimes abrupt changes, when degeneracies are present).

Now that we have illustrated what exactly we have constructed, let's use the SOM to predict the redshift distribution for a set of photometric objects.  We will make a simple cut in spectroscopic redshift to create a compact redshift bin.  In more realistic circumstances we would likely be using color cuts or photometric redshift estimates to define our test bin(s).  We will cut our photometric sample to only include galaxies in 0.2<specz<0.5.

We will need to trim both our spec-z set to i<23.5 to match our trained SOM:

In [ ]:
testfile = find_rail_file('examples_data/testdata/test_dc2_training_9816.hdf5')
data = tables_io.read(testfile)['photometry']
mask = ((data['redshift'] > 0.2) & (data['redshift']<0.5))
brightmask = ((mask) & (data['mag_i_lsst']<23.5))
trim_data = {}
bright_data = {}
for key in data.keys():
    trim_data[key] = data[key][mask]
    bright_data[key] = data[key][brightmask]
trimdict = dict(photometry=trim_data)
brightdict = dict(photometry=bright_data)
# set up data as data handles
test_data = Hdf5Handle("tomo_bin", data=trimdict)
bright_data = Hdf5Handle("bright_bin",data=brightdict)

In [ ]:
specfile = find_rail_file("examples_data/testdata/test_dc2_validation_9816.hdf5")
spec_data = tables_io.read(specfile)['photometry']
smask = (spec_data['mag_i_lsst'] <23.5)
trim_spec = {}
for key in spec_data.keys():
    trim_spec[key] = spec_data[key][smask]
trim_dict = dict(photometry=trim_spec)
# set up data as data handles
spec_data = Hdf5Handle("spec_data",data=trim_dict)

Note that we have removed the 'photometry' group, we will specify the `phot_groupname` as "" in the parameters below.

## Gold weight: combining multiple SOM realizations

Following Stölzner et al. 2025 ([arXiv:2503.19440](https://arxiv.org/abs/2503.19440), Sect. 4), a SOM cell is classified **gold** in a given realization if it contains at least one calibrating spec-z galaxy (i.e. it is a "covered" cell in RAIL's terminology, the opposite of the cells written to `uncovered_cell_file`); photometric galaxies that land in a non-gold cell have no representative spec-z and would previously have been discarded by a hard gold-class cut.

For each of our `N_REALIZATIONS` trained SOMs we:
1. find the best-matching cell for every spec-z (calibration) galaxy, and flag all cells that contain at least one such galaxy as "gold";
2. find the best-matching cell for every photometric galaxy, and record its binary gold classification $g_{i,j}$ (1 if its cell is gold in realization $j$, else 0).

The **gold weight** of galaxy $i$ is then the fraction of realizations in which it was classified gold:

$$W^{\rm gold}_i = \frac{1}{N_{\rm repl}}\sum_{j=1}^{N_{\rm repl}} g_{i,j}.$$

We compute this for both of our photometric samples (`test_data` and `bright_data`) and store it as a new `gold_weight` column, which is added to the data and will be used as the photometric weight (`phot_weightcol`) in the SOM summarizer below. We also compute it for the SOM **training set** (`training_data`), which is used only for the Figure 3-style plot below. For comparison, we also compute the hard 0/1 `gold_class` from `model_goldclass` (the dedicated SOM trained earlier, independent of this `N_REALIZATIONS` ensemble) for the two photometric samples -- purely for inspection here, since the actual gold-class N(z) further down is built by `SOMocluSummarizer`'s own internal logic on `model_goldclass`, not from this column, so that it reproduces the traditional single-SOM gold-class calculation exactly.

In [ ]:
def compute_gold_class(som_model, spec_dict, phot_dict):
    """Gold classification of `phot_dict` galaxies for a single SOM realization.

    A SOM cell is 'gold' if at least one calibrating (spec-z) galaxy from
    `spec_dict` has that cell as its best-matching unit. Returns the binary
    gold class of each photometric galaxy, the gold cell mask (shape
    n_columns x n_rows), and the raveled photometric pixel index of each
    galaxy (useful for the Figure 3-style plots below).
    """
    som = som_model['som']
    som_usecols = som_model['usecols']
    som_ref_column = som_model['ref_column']
    som_column_usage = som_model['column_usage']
    som_n_rows = som_model['n_rows']
    som_n_columns = som_model['n_columns']

    spec_colors = _computemagcolordata(spec_dict, som_ref_column, som_usecols, som_column_usage)
    spec_bmu = get_bmus(som, spec_colors)
    spec_pix = np.ravel_multi_index((spec_bmu[:, 0], spec_bmu[:, 1]), (som_n_columns, som_n_rows))
    gold_cell_mask = np.bincount(spec_pix, minlength=som_n_columns * som_n_rows) > 0

    phot_colors = _computemagcolordata(phot_dict, som_ref_column, som_usecols, som_column_usage)
    phot_bmu = get_bmus(som, phot_colors)
    phot_pix = np.ravel_multi_index((phot_bmu[:, 0], phot_bmu[:, 1]), (som_n_columns, som_n_rows))

    gold_class = gold_cell_mask[phot_pix].astype(float)
    return gold_class, gold_cell_mask.reshape(som_n_columns, som_n_rows), phot_pix

In [ ]:
%%time
spec_dict = spec_data.data['photometry']
test_dict = test_data.data['photometry']
bright_dict = bright_data.data['photometry']
train_dict = training_data.data['photometry']

n_test = len(test_dict[usecols[0]])
n_bright = len(bright_dict[usecols[0]])
n_train_gold = len(train_dict[usecols[0]])

gold_class_test = np.zeros((N_REALIZATIONS, n_test))
gold_class_bright = np.zeros((N_REALIZATIONS, n_bright))
gold_class_train = np.zeros((N_REALIZATIONS, n_train_gold))

# diagnostics from the fiducial (first) realization, used for the Figure 3-style plots below
fiducial_gold_cell_mask = None
fiducial_phot_pix_train = None

for j, som_model_j in enumerate(som_models):
    gold_class_test[j], gold_mask_j, _ = compute_gold_class(som_model_j, spec_dict, test_dict)
    gold_class_bright[j], _, _ = compute_gold_class(som_model_j, spec_dict, bright_dict)
    gold_class_train[j], _, phot_pix_j_train = compute_gold_class(som_model_j, spec_dict, train_dict)
    if j == 0:
        fiducial_gold_cell_mask = gold_mask_j
        fiducial_phot_pix_train = phot_pix_j_train
    print(f"Realization {j + 1}/{N_REALIZATIONS}: "
          f"{int(gold_mask_j.sum())}/{gold_mask_j.size} SOM cells classified gold")

In [ ]:
gold_weight_test = gold_class_test.mean(axis=0)
gold_weight_bright = gold_class_bright.mean(axis=0)
gold_weight_train = gold_class_train.mean(axis=0)

# add the gold weight to the photometric data as a new per-galaxy entry
test_data.data['photometry']['gold_weight'] = gold_weight_test
bright_data.data['photometry']['gold_weight'] = gold_weight_bright
training_data.data['photometry']['gold_weight'] = gold_weight_train

# hard gold class (KiDS-1000-style cut), from the dedicated model_goldclass SOM
# trained above. This column is stored for inspection/plotting (e.g. Figure 3
# below and the gold fraction printed here), but -- unlike gold_weight -- it is
# NOT passed to the SOMocluSummarizer as a phot_weightcol further down. That
# summarizer call instead uses model_goldclass with no weight column at all,
# matching the traditional single-SOM approach, so that the gold-class N(z) relies on
# SOMocluSummarizer's own internal covered/uncovered-cell logic rather than a
# hand-rolled reimplementation of it (which is not guaranteed to agree exactly,
# e.g. it does not replicate the summarizer's non-detection replacement step).
goldclass_col_test, goldclass_cell_mask, _ = compute_gold_class(model_goldclass, spec_dict, test_dict)
goldclass_col_bright, _, _ = compute_gold_class(model_goldclass, spec_dict, bright_dict)
test_data.data['photometry']['gold_class'] = goldclass_col_test
bright_data.data['photometry']['gold_class'] = goldclass_col_bright

print(f"full sample: mean gold weight = {gold_weight_test.mean():.3f}, "
      f"fraction with weight==1: {(gold_weight_test == 1).mean():.3f}, "
      f"fraction with weight==0: {(gold_weight_test == 0).mean():.3f}")
print(f"bright sample: mean gold weight = {gold_weight_bright.mean():.3f}, "
      f"fraction with weight==1: {(gold_weight_bright == 1).mean():.3f}, "
      f"fraction with weight==0: {(gold_weight_bright == 0).mean():.3f}")
print(f"training set: mean gold weight = {gold_weight_train.mean():.3f}, "
      f"fraction with weight==1: {(gold_weight_train == 1).mean():.3f}, "
      f"fraction with weight==0: {(gold_weight_train == 0).mean():.3f}")
print(f"full sample: dedicated gold-class SOM gold fraction = {goldclass_col_test.mean():.3f}")
print(f"bright sample: dedicated gold-class SOM gold fraction = {goldclass_col_bright.mean():.3f}")

## Figure-3-style plot: gold class vs. gold weight

Reproducing Figure 3 of Stölzner et al. 2025 ([arXiv:2503.19440](https://arxiv.org/pdf/2503.19440)), we show, on our fiducial SOM grid:
- **left:** the true mean redshift of the calibration galaxies in each cell;
- **centre:** the binary gold class of each cell under a *single* SOM realization (1 = contains calibrating spec-z, 0 = does not);
- **right:** the gold weight of each cell after combining all `N_REALIZATIONS` SOM trainings (the average `gold_weight` of the SOM **training-set** galaxies whose best-matching cell, under the fiducial SOM, is that cell).

As in the paper's Figure 3, in the right-hand (gold-weight) panel we **highlight with an orange border** the cells that have zero gold class in the single fiducial realization (i.e. the cells shown as 0/non-gold in the centre panel). This makes it easy to see that some of these "stochastically non-gold" cells nonetheless end up with substantial gold weight once all `N_REALIZATIONS` are combined, illustrating why a continuous weight is preferable to a hard cut.

In [ ]:
from matplotlib.patches import Rectangle

def highlight_cells(ax, mask, grid_type='hexagonal', edgecolor='orange', lw=2.5):
    """Draw a colored outline around cells where `mask` is True.

    `mask` must have the same shape and orientation as the array passed to
    `plot_som` (i.e. already transposed), so that outlines land on the
    correct cells. This mirrors plot_som's own cell-position geometry
    (rail.estimation.algos.somoclu_som) without modifying that function.
    """
    som_dim = mask.shape[0]
    if grid_type == 'rectangular':
        for i in range(mask.shape[0]):
            for j in range(mask.shape[1]):
                if mask[i, j]:
                    rect = Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor='none',
                                      edgecolor=edgecolor, lw=lw, zorder=5)
                    ax.add_patch(rect)
    else:
        yy, xx = np.meshgrid(np.arange(som_dim), np.arange(som_dim))
        shift = np.zeros(som_dim)
        shift[::2] = -0.5
        xx = xx + shift
        for i in range(mask.shape[0]):
            for j in range(mask.shape[1]):
                if mask[i, j]:
                    wy = yy[(i, j)] * np.sqrt(3) / 2
                    outline = RegularPolygon((xx[(i, j)], wy), numVertices=6,
                                              radius=1 / np.sqrt(3), facecolor='none',
                                              edgecolor=edgecolor, lw=lw, zorder=5)
                    ax.add_patch(outline)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(24, 8))

# left panel: true mean redshift per cell (fiducial SOM, calibration/spec sample)
spec_colors0 = _computemagcolordata(spec_dict, ref_column_name, usecols, column_usage)
spec_bmu0 = get_bmus(SOM, spec_colors0)
spec_meanz = np.zeros((n_columns, n_rows))
spec_occ = np.zeros((n_columns, n_rows))
np.add.at(spec_meanz, (spec_bmu0[:, 0], spec_bmu0[:, 1]), spec_dict['redshift'])
np.add.at(spec_occ, (spec_bmu0[:, 0], spec_bmu0[:, 1]), 1)
with np.errstate(invalid='ignore'):
    spec_meanz = np.where(spec_occ > 0, spec_meanz / np.maximum(spec_occ, 1), np.nan)

plot_som(axes[0], spec_meanz.T, grid_type=grid_type, colormap=cm.coolwarm, cbar_name='true mean redshift')
axes[0].set_title('mean redshift per cell\n(fiducial SOM realization)', fontsize=13)

# centre panel: gold class under the single fiducial realization
gold_class_map = np.where(fiducial_gold_cell_mask, 1.0, 0.0)
plot_som(axes[1], gold_class_map.T, grid_type=grid_type, colormap=cm.coolwarm,
         cbar_name='gold class (0/1)', vmin=0, vmax=1)
axes[1].set_title('gold class\n(single SOM realization)', fontsize=13)

# right panel: gold weight after N_REALIZATIONS SOM trainings, on the fiducial
# grid, evaluated for the SOM TRAINING SET (rather than the photometric test sample)
cell_sum = np.zeros(n_columns * n_rows)
cell_cnt = np.zeros(n_columns * n_rows)
np.add.at(cell_sum, fiducial_phot_pix_train, gold_weight_train)
np.add.at(cell_cnt, fiducial_phot_pix_train, 1)
gold_weight_map = np.full(n_columns * n_rows, np.nan)
covered = cell_cnt > 0
gold_weight_map[covered] = cell_sum[covered] / cell_cnt[covered]
gold_weight_map = gold_weight_map.reshape(n_columns, n_rows)

plot_som(axes[2], gold_weight_map.T, grid_type=grid_type, colormap=cm.coolwarm,
         cbar_name='gold weight', vmin=0, vmax=1)
axes[2].set_title(f'gold weight\n(after {N_REALIZATIONS} SOM realizations,\ntraining set)', fontsize=13)

# highlight, in orange, the cells that have zero gold class in the single
# fiducial realization -- same cells shown as 0 in the centre panel
highlight_cells(axes[2], (~fiducial_gold_cell_mask).T, grid_type=grid_type)

plt.tight_layout()

## Building the final redshift distribution with the gold weight

As before, let us specify our initialization params for the SomocluSOMSummarizer stage, including:

- `model`: the fiducial trained SOM model (we pass the model object directly rather than a filename)
- `hdf5_groupname` (str): hdf5 group for our photometric data (in our case "")
- `objid_name` (str): string specifying the name of the ID column, if present photom data, will be written out to cellid_output file
- `spec_groupname` (str): hdf5 group for the spectroscopic data
- `nzbins` (int): number of bins to use in our histogram ensemble
- `nsamples` (int): number of bootstrap samples to generate
- `output` (str): name of the output qp file with N samples
- `single_NZ` (str): name of the qp file with fiducial distribution
- `uncovered_cell_file` (str): name of hdf5 file containing a list of all of the cells with phot data but no spec-z objects
- **`phot_weightcol` (str): set to `'gold_weight'`, the KiDS-Legacy per-galaxy weight computed above.** Photometric galaxies are still all included, but those that are only intermittently gold across SOM realizations are down-weighted in the histogram sum rather than being hard-cut.

In [ ]:
summ_dict = dict(model=model, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, nsamples=25,
                 output='KL_SOM_ensemble.hdf5', single_NZ='KL_fiducial_SOMoclu_NZ.hdf5',
                 uncovered_cell_file='KL_all_uncovered_cells.hdf5',
                 objid_name='id',
                 cellid_output='KL_output_cellIDs.hdf5',
                 phot_weightcol='gold_weight')

Now let's initialize and run the summarizer.  One feature of the SOM: if any SOM cells contain photometric data but do not contain any redshifts values in the spectroscopic set, then no reasonable redshift estimate for those objects is defined, and they are skipped.  The method currently prints the indices of uncovered cells, we may modify the algorithm to actually output the uncovered galaxies in a separate file in the future.

In [ ]:
som_summarizer = SOMocluSummarizer.make_stage(name='SOMoclu_summarizer', **summ_dict)

In [ ]:
output_dict = som_summarizer.summarize(test_data, spec_data)

**Comparison with the KiDS-1000-style hard gold-class cut.** To quantify what the continuous gold weight buys us, let's also build the N(z) using the **dedicated, separately-trained `model_goldclass` SOM** from above -- not `model` (the gold-weight ensemble's fiducial SOM). We pass **no `phot_weightcol`** here, just like the traditional single-SOM gold-class approach: `SOMocluSummarizer` already excludes photometric galaxies from SOM cells with no calibrating spec-z internally, which *is* the gold-class cut, so this reproduces that calculation exactly rather than approximating it with a hand-rolled weight column.

In [ ]:
summ_dict_goldclass = dict(model=model_goldclass, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, nsamples=25,
                 output='KL_goldclass_SOM_ensemble.hdf5', single_NZ='KL_goldclass_fiducial_SOMoclu_NZ.hdf5',
                 uncovered_cell_file='KL_goldclass_all_uncovered_cells.hdf5',
                 objid_name='id',
                 cellid_output='KL_goldclass_output_cellIDs.hdf5')
som_summarizer_goldclass = SOMocluSummarizer.make_stage(name='SOMoclu_summarizer_goldclass', **summ_dict_goldclass)

In [ ]:
output_dict_goldclass = som_summarizer_goldclass.summarize(test_data, spec_data)

Let's open the fiducial N(z) file, plot it, and see how it looks, and compare it to the true tomographic bin file:

In [ ]:
fid_ens = qp.read("KL_fiducial_SOMoclu_NZ.hdf5")
goldclass_fid_ens = qp.read("KL_goldclass_fiducial_SOMoclu_NZ.hdf5")

In [ ]:
def get_cont_hist(data, bins):
    hist, bin_edge = np.histogram(data, bins=bins, density=True)
    return hist, (bin_edge[1:]+bin_edge[:-1])/2

In [ ]:
test_nz_hist, zbin = get_cont_hist(test_data.data['photometry']['redshift'], np.linspace(0,3,101))
som_nz_hist = np.squeeze(fid_ens.pdf(zbin))
goldclass_nz_hist = np.squeeze(goldclass_fid_ens.pdf(zbin))

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12,8))
ax.set_xlabel("redshift", fontsize=15)
ax.set_ylabel("N(z)", fontsize=15)
ax.plot(zbin, test_nz_hist, label='True N(z)')
ax.plot(zbin, som_nz_hist, label='SOM N(z), gold-weighted')
ax.plot(zbin, goldclass_nz_hist, label='SOM N(z), gold-class cut (single realization)', ls='--')
plt.legend()

Seems fine, roughly the correct redshift range for the lower redshift peak, but a few secondary peaks at large z tail.  What if we try the bright dataset that we made?

In [ ]:
bright_dict_summ = dict(model=model, hdf5_groupname='photometry',
                   spec_groupname='photometry', nzbins=101, nsamples=25,
                   output='KL_BRIGHT_SOMoclu_ensemble.hdf5', single_NZ='KL_BRIGHT_fiducial_SOMoclu_NZ.hdf5',
                   uncovered_cell_file="KL_BRIGHT_uncovered_cells.hdf5",
                   objid_name='id',
                   cellid_output='KL_BRIGHT_output_cellIDs.hdf5',
                   phot_weightcol='gold_weight')
bright_summarizer = SOMocluSummarizer.make_stage(name='bright_summarizer', **bright_dict_summ)

bright_dict_summ_goldclass = dict(model=model_goldclass, hdf5_groupname='photometry',
                   spec_groupname='photometry', nzbins=101, nsamples=25,
                   output='KL_goldclass_BRIGHT_SOMoclu_ensemble.hdf5', single_NZ='KL_goldclass_BRIGHT_fiducial_SOMoclu_NZ.hdf5',
                   uncovered_cell_file="KL_goldclass_BRIGHT_uncovered_cells.hdf5",
                   objid_name='id',
                   cellid_output='KL_goldclass_BRIGHT_output_cellIDs.hdf5')
bright_summarizer_goldclass = SOMocluSummarizer.make_stage(name='bright_summarizer_goldclass', **bright_dict_summ_goldclass)

In [ ]:
bright_output_dict = bright_summarizer.summarize(bright_data, spec_data)

In [ ]:
bright_output_dict_goldclass = bright_summarizer_goldclass.summarize(bright_data, spec_data)

In [ ]:
bright_fid_ens = qp.read("KL_BRIGHT_fiducial_SOMoclu_NZ.hdf5")
bright_goldclass_fid_ens = qp.read("KL_goldclass_BRIGHT_fiducial_SOMoclu_NZ.hdf5")

In [ ]:
bright_nz_hist, zbin = get_cont_hist(bright_data.data['photometry']['redshift'], np.linspace(0,3,101))
bright_som_nz_hist = np.squeeze(bright_fid_ens.pdf(zbin))
bright_goldclass_nz_hist = np.squeeze(bright_goldclass_fid_ens.pdf(zbin))

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12,8))
ax.set_xlabel("redshift", fontsize=15)
ax.set_ylabel("N(z)", fontsize=15)
ax.plot(zbin, bright_nz_hist, label='True N(z), bright')
ax.plot(zbin, bright_som_nz_hist, label='SOM N(z), bright, gold-weighted')
ax.plot(zbin, bright_goldclass_nz_hist, label='SOM N(z), bright, gold-class cut (single realization)', ls='--')
plt.legend()

Looks better, we've eliminated the secondary peak. Now, SOMs are a bit touchy to train, and are highly dependent on the dataset used to train them.  This demo used a relatively small dataset (~150,000 DC2 galaxies from one healpix pixel) to train the SOM, and even smaller photometric and spectroscopic datasets of 10,000 and 20,000 galaxies.  We should expect slightly better results with more data, at least in cells where the spectroscopic data is representative.

However, there is a caveat that SOMs are not guaranteed to converge, and are very sensitive to both the input data and tunable parameters of the model.  So, users should do some verification tests before trusting the SOM is going to give accurate results.  The gold-weighting scheme introduced here mitigates (but does not eliminate) the sensitivity of the *gold/non-gold classification* to any single SOM training; the underlying SOM itself is still just one realization.

Finally, let's load up our bootstrap ensembles and overplot N(z) of bootstrap samples:

In [ ]:
boot_ens = qp.read("KL_BRIGHT_SOMoclu_ensemble.hdf5")

In [ ]:
fig, ax=plt.subplots(1,1,figsize=(8, 8))
ax.set_xlim((0,1))
ax.set_xlabel("redshift", fontsize=20)
ax.set_ylabel("N(z)", fontsize=20)

ax.plot(zbin, bright_nz_hist, lw=2, label='True N(z)', color='C1', zorder=1)
ax.plot(zbin, bright_som_nz_hist, lw=2, label='SOM mean N(z), gold-weighted', color='k', zorder=2)

for i in range(boot_ens.npdf):
    pdf = np.squeeze(boot_ens[i].pdf(zbin))
    if i == 0:
        ax.plot(zbin, pdf, color='C2',zorder=0, lw=2, alpha=0.5, label='SOM N(z) samples')
    else:
        ax.plot(zbin, pdf, color='C2',zorder=0, lw=2, alpha=0.5)
plt.legend(fontsize=20)
plt.xlim(0, 1.5)

plt.xticks(fontsize=18)
plt.yticks(fontsize=18)


## Quantitative metrics

Let's look at how we've done at estimating the mean redshift and "width" (via standard deviation) of our tomographic bin compared to the true redshift and "width" for both our "full" sample and "bright" i<23.5 samples.

We compute these statistics for **both** calibration schemes -- the continuous, multi-realization gold weight and the single-realization, hard gold-class cut -- so that we can directly compare how much (if at all) the gold-weighting scheme changes the bias and scatter of the recovered mean redshift and width. Further down we histogram these bootstrap-to-bootstrap biases directly for the two schemes.

In [ ]:
full_ens = qp.read("KL_SOM_ensemble.hdf5")
full_means = full_ens.mean().flatten()
full_stds = full_ens.std().flatten()
true_full_mean = np.mean(test_data.data['photometry']['redshift'])
true_full_std = np.std(test_data.data['photometry']['redshift'])

In [ ]:
# gold-class (hard-cut) ensemble, for comparison with the gold-weighted result above
goldclass_ens = qp.read("KL_goldclass_SOM_ensemble.hdf5")
goldclass_means = goldclass_ens.mean().flatten()
goldclass_stds = goldclass_ens.std().flatten()

Let's check the accuracy and precision of mean readshift:

In [ ]:
print("The mean redshift of the SOM ensemble is: "+str(round(np.mean(full_means),4)) + '+-' + str(round(np.std(full_means),4)))
print("The mean redshift of the real data is: "+str(round(true_full_mean,4)))
print("The bias of mean redshift is:"+str(round(np.mean(full_means)-true_full_mean,4)) + '+-' + str(round(np.std(full_means),4)))

In [ ]:
# compare the gold-weight (continuous, multi-realization) and gold-class (hard-cut,
# single-realization) calibration schemes for the full sample
print("[gold-weight]  mean redshift: " + str(round(np.mean(full_means), 4)) + '+-' + str(round(np.std(full_means), 4))
      + "   bias: " + str(round(np.mean(full_means) - true_full_mean, 4)) + '+-' + str(round(np.std(full_means), 4)))
print("[gold-class]   mean redshift: " + str(round(np.mean(goldclass_means), 4)) + '+-' + str(round(np.std(goldclass_means), 4))
      + "   bias: " + str(round(np.mean(goldclass_means) - true_full_mean, 4)) + '+-' + str(round(np.std(goldclass_means), 4)))

In [ ]:
bright_means = boot_ens.mean().flatten()
bright_stds = boot_ens.std().flatten()
true_bright_mean = np.mean(bright_data.data['photometry']['redshift'])
true_bright_std = np.std(bright_data.data['photometry']['redshift'])

In [ ]:
# gold-class (hard-cut) ensemble for the bright sample, for comparison
goldclass_bright_ens = qp.read("KL_goldclass_BRIGHT_SOMoclu_ensemble.hdf5")
goldclass_bright_means = goldclass_bright_ens.mean().flatten()
goldclass_bright_stds = goldclass_bright_ens.std().flatten()

In [ ]:
print("The mean redshift of the SOM ensemble is: "+str(round(np.mean(bright_means),4)) + '+-' + str(round(np.std(bright_means),4)))
print("The mean redshift of the real data is: "+str(round(true_bright_mean,4)))
print("The bias of mean redshift is:"+str(round(np.mean(bright_means)-true_bright_mean, 4)) + '+-' + str(round(np.std(bright_means),4)))

In [ ]:
# compare the gold-weight and gold-class calibration schemes for the bright sample
print("[gold-weight]  mean redshift: " + str(round(np.mean(bright_means), 4)) + '+-' + str(round(np.std(bright_means), 4))
      + "   bias: " + str(round(np.mean(bright_means) - true_bright_mean, 4)) + '+-' + str(round(np.std(bright_means), 4)))
print("[gold-class]   mean redshift: " + str(round(np.mean(goldclass_bright_means), 4)) + '+-' + str(round(np.std(goldclass_bright_means), 4))
      + "   bias: " + str(round(np.mean(goldclass_bright_means) - true_bright_mean, 4)) + '+-' + str(round(np.std(goldclass_bright_means), 4)))

**Histogram comparison of bias and scatter.** Rather than plotting individual bootstrap draws as vertical lines, let's histogram the two quantities that matter for calibration: the **bias** in the recovered mean redshift (each bootstrap draw's mean minus the true mean) and the **bias in the width** (each bootstrap draw's std minus the true std, which captures whether a scheme over/under-estimates the scatter of the tomographic bin). For each sample (full, bright) we overlay the gold-weight and gold-class distributions of these two quantities across the `nsamples` bootstrap draws, so a systematic shift or a wider spread for one scheme relative to the other is immediately visible, and a perfectly unbiased scheme would have its histogram centred on zero (dashed line).

In [ ]:
full_mean_bias = full_means - true_full_mean
goldclass_mean_bias = goldclass_means - true_full_mean
full_std_bias = full_stds - true_full_std
goldclass_std_bias = goldclass_stds - true_full_std

bright_mean_bias = bright_means - true_bright_mean
goldclass_bright_mean_bias = goldclass_bright_means - true_bright_mean
bright_std_bias = bright_stds - true_bright_std
goldclass_bright_std_bias = goldclass_bright_stds - true_bright_std

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 12))

def bias_hist(ax, gw_vals, gc_vals, xlabel, title):
    bins = np.linspace(min(gw_vals.min(), gc_vals.min()),
                        max(gw_vals.max(), gc_vals.max()), 12)
    ax.hist(gw_vals, bins=bins, alpha=0.6, color='C0', label='gold-weight')
    ax.hist(gc_vals, bins=bins, alpha=0.6, color='C1', label='gold-class')
    ax.axvline(0, color='k', ls='--', lw=1.5)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('count', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=10)

bias_hist(axes[0, 0], full_mean_bias, goldclass_mean_bias,
          'bias in mean redshift (bootstrap - true)', 'full sample: bias in mean redshift')
bias_hist(axes[0, 1], full_std_bias, goldclass_std_bias,
          'bias in width/std (bootstrap - true)', 'full sample: scatter (bias in width)')
bias_hist(axes[1, 0], bright_mean_bias, goldclass_bright_mean_bias,
          'bias in mean redshift (bootstrap - true)', 'bright sample: bias in mean redshift')
bias_hist(axes[1, 1], bright_std_bias, goldclass_bright_std_bias,
          'bias in width/std (bootstrap - true)', 'bright sample: scatter (bias in width)')

plt.tight_layout()

For both cases, the mean redshifts seem to be pretty precise and accurate (bright sample seems more precise). For the full sample, the SOM N(z) are slightly wider, while for the bright sample the widths are also fairly accurate.
For both cases, the errors in mean redshift are at levels of ~0.005, close to the tolerance for cosmological analysis. However, we have not consider the photometric error in magnitudes and colors, as well as additional color selections. Our sample is also limited. This demo only serves as a preliminary implementation of the KiDS-Legacy gold-weighted SOM calibration in RAIL.

Compared to the original (unweighted, single-SOM, hard gold-cut) approach, the gold-weight scheme trades a small amount of extra compute (training `N_REALIZATIONS` SOMs instead of one) for a photometric weighting that is less sensitive to the randomness of any single SOM training, following Stölzner et al. 2025 ([arXiv:2503.19440](https://arxiv.org/abs/2503.19440)) and Wright et al. 2025 ([arXiv:2503.19441](https://arxiv.org/abs/2503.19441)).

The gold-class vs. gold-weight comparison above (both in the N(z) plots and in the mean/std bias metrics) shows how much of a difference this makes in practice for this particular demo dataset: any large disagreement between the two schemes indicates that the hard, single-realization gold cut was sensitive to the specific stochastic SOM training used, exactly the effect that motivated the KiDS-Legacy move to a continuous gold weight.